<a href="https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Home%20Task/Home-Task-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗄️ The Analytical Language of John Wilkins

> *"These ambiguities, redundancies, and deficiencies recall those attributed by Dr. Franz Kuhn to a certain Chinese encyclopedia called the *Celestial Emporium of Benevolent Knowledge*. On those remote pages it is written that animals are divided into (a) those that belong to the Emperor, (b) embalmed ones, (c) those that are trained, (d) suckling pigs, (e) mermaids, (f) fabulous ones, (g) stray dogs, (h) those that are included in this classification..."*
>
> — Jorge Luis Borges, *The Analytical Language of John Wilkins*

## The story

In the seventeenth century a churchman named John Wilkins set out to build a perfect language — one in which the very *spelling* of a word would declare the nature of the thing it named. Each animal would be filed under a rigorous tree of yes-and-no distinctions: beast or fish, winged or finned, tame or wild, until the creature stood alone at the end of a single branch, named by the path that led to it.

The scheme failed, as all such schemes fail. But somewhere a clerk kept building it anyway. He bound every animal of the world into a great **Cabinet of Distinctions** — and then, before the index could be written, he died. The drawers remain. Each holds one creature behind a small brass grille, and the creature will not say its name. It will only answer **yes** or **no** to questions about its own nature.

The Cabinet has come to you with its labels lost. Two lists survived in the clerk's hand:

- `animals_pool.txt` — every creature filed in the Cabinet (~1,400 entries).
- `questions_pool.txt` — every distinction the clerk thought to draw (~500 yes/no questions).

Open a drawer. Ask your distinctions. Find the path that names the beast.

## Your task

Each hidden creature sits inside a sealed oracle called an **Interactor** — a brass grille over a drawer, holding one animal. You cannot see it. You may put to it one of two kinds of question:

| Call | Returns | What you are asking |
|---|---|---|
| `interactor.ask(question)` | `"yes"` or `"no"` | A yes/no question about the hidden animal. The question must be a line from `questions_pool.txt`. |
| `interactor.guess(animal)` | `"correct"` or `"wrong"` | "Is this the hidden animal?" `"correct"` ends the row. The animal must be a word from `animals_pool.txt`. |

Each question in the pool refers to the creature generically — *"is it a mammal?"*, *"does it live in water?"*, *"can it fly?"* — and the oracle answers about whichever animal is hidden in that drawer.

If you submit a question or animal not in the relevant pool, the oracle refuses without spending its strength: a `ValueError` is raised and your budget is unchanged. Typos cost nothing.

Each drawer will entertain at most **fifteen questions** before the grille falls shut.

### Scoring

For each creature:

```
score = max(0, (1 if you ever guessed correctly else 0) - 0.02 × queries_used)
```

- Correct guess on question 1 → 0.98
- Correct guess on question 5 → 0.90
- Correct guess on question 15 → 0.70
- Never correct → 0

Your score is the mean across all creatures in a test set. Tune on `dev`, then run the final cell to get your **`test1`** score — that summary table is what you submit a screenshot of. The organizers keep a second, **hidden** test set for official grading, so a solution that genuinely deduces (rather than overfits `dev`/`test1`) is what scores well.

## The oracle

In plain language, the oracle inside each Interactor is a local language model — by default, `Qwen/Qwen2.5-3B-Instruct`. When you call `ask(question)` it prompts the model with:

```
You are answering a question about one specific animal.
The animal is: <hidden animal>.
Answer with a single word, yes or no.
Question: <question>
```

at temperature 0, parses the first word of the reply, and returns `"yes"` or `"no"` to your code.

The model is deterministic (the same `(animal, question)` pair always gives the same answer) and runs entirely inside the Interactor. You are free to run the same model in your own code to **predict** what it will say without spending the oracle's strength — that is a large part of what makes a clever solution. Note the oracle answers from the model's *beliefs* about the animal, which are usually right but not infallible; a good solution is robust to the occasional surprising answer.

## Step 1: Setup

The dataset and helper code (`interactor.py`, `evaluate.py`, the two pools, the dev/test CSVs) live in the shared **`IOAI-2026/AnimalDeduction/dataset`** Drive folder. The cell below just downloads them into Colab — no sign-in, no shortcuts, just run it. Use a **GPU** runtime: *Runtime → Change runtime type → T4* (free tier is enough).

In [1]:
!pip install -q gdown transformers accelerate

import os, sys
from pathlib import Path
import gdown

# Dataset + helper code live in the shared IOAI-2026/AnimalDeduction/dataset folder
# (public link). Download locally and import from there — no sign-in needed.
LOCAL_DIR = Path('content/animaldeduction')
if not LOCAL_DIR.exists() or not any(LOCAL_DIR.iterdir()):
    gdown.download_folder(id='1YheHvGfQw5YUa7MjdUF0hQC4sdtLZ5UC',
                          output=str(LOCAL_DIR), quiet=True, use_cookies=False)

sys.path.insert(0, str(LOCAL_DIR))
os.chdir(LOCAL_DIR)
print('Working directory:', os.getcwd())
print('Files:', sorted(p.name for p in LOCAL_DIR.iterdir()))


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Working directory: c:\Users\raian\source\repos\AI\IOAI_prep\ioai\2026\problem_3\content\animaldeduction
Files: ['animals_pool.txt', 'dev.csv', 'evaluate.py', 'interactor.py', 'questions_pool.txt', 'test1.csv']


## Step 2: Load data and try the oracle

The `Interactor` owns the hidden gold animal and runs a local LLM (Qwen 2.5 3B Instruct by default) to answer yes/no questions about it. The first `Interactor(...)` instantiation triggers the LLM download (~6 GB on first run, takes 30-60 s on T4). Every subsequent Interactor reuses the same LLM that's already loaded in memory.

In [2]:
import random
import numpy as np
import pandas as pd
import torch

from interactor import Interactor
from evaluate import evaluate, load_pools

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

animals_pool, questions_pool = load_pools()
print(f'animals_pool size:   {len(animals_pool):>6}  (e.g. {animals_pool[:5]})')
print(f'questions_pool size: {len(questions_pool):>6}  (e.g. {questions_pool[:3]})')

# Sanity probe: create one Interactor and ask two questions about an octopus.
probe = Interactor(gold_animal='octopus', animals_pool=animals_pool, questions_pool=questions_pool)
print("\nask('is it a mammal?')      ->", probe.ask('is it a mammal?'))
print("ask('does it live in water?') ->", probe.ask('does it live in water?'))
print('Queries used:', probe.queries_used, '/', probe.budget)

Device: cuda
animals_pool size:     1472  (e.g. ['lion', 'tiger', 'leopard', 'snow leopard', 'cheetah'])
questions_pool size:    559  (e.g. ['is it a mammal?', 'is it a bird?', 'is it a reptile?'])
  [interactor] loading Qwen/Qwen2.5-3B-Instruct on cuda...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [interactor] LLM ready.

ask('is it a mammal?')      -> no
ask('does it live in water?') -> yes
Queries used: 2 / 15


## Step 3: Solution interface

Your solution is a class with two methods:

- `__init__(self, animals_pool, questions_pool)` — runs once. Load models, precompute tables, etc.
- `solve(self, interactor)` — runs once per test row. Use the oracle to identify the hidden animal.

Inside `solve`, you have:

```
interactor.ask(question)      -> 'yes' or 'no'        (question must be in questions_pool)
interactor.guess(animal)      -> 'correct' or 'wrong' (animal must be in animals_pool)
interactor.is_done()          -> True after a correct guess or budget exhausted
interactor.remaining_budget() -> int
```

**Scoring per row**: `score = max(0, (1 if you ever guess correctly else 0) - 0.02 * total_queries)`.

Budget is **15 questions** per row. Information theory: `log₂(1400) ≈ 10.5 bits`, each yes/no answer is at most 1 bit — so ~11 well-chosen questions plus 1 final guess fit the budget, *if* each question splits the remaining candidates in half. Most don't: *"does it have a backbone?"* sounds decisive but the model calls a great many creatures vertebrates. A good question splits the *remaining* candidates roughly in half, given everything you have already learned — so the right next question depends on the answers so far.

### Baseline: random guessing (the floor)

Ignores `ask()` entirely. Just guesses random animals until the budget runs out. Expected score: ~0 (15 random guesses out of ~1,400 candidates ≈ 1% solve rate). Any reasonable solution needs to beat this by a lot.

In [3]:
class RandomBaseline:
    def __init__(self, animals_pool, questions_pool, seed=0):
        self.animals_pool = animals_pool
        self.questions_pool = questions_pool
        self.rng = random.Random(seed)

    def solve(self, interactor):
        guessed = set()
        while not interactor.is_done():
            cand = self.rng.choice(self.animals_pool)
            while cand in guessed:
                cand = self.rng.choice(self.animals_pool)
            guessed.add(cand)
            interactor.guess(cand)

baseline_results = evaluate(RandomBaseline(animals_pool, questions_pool), 'dev.csv')

  25/150 rows  mean_score=0.0000  (0.0s)
  50/150 rows  mean_score=0.0156  (0.0s)
  75/150 rows  mean_score=0.0219  (0.0s)
  100/150 rows  mean_score=0.0242  (0.0s)
  125/150 rows  mean_score=0.0194  (0.0s)
  150/150 rows  mean_score=0.0161  (0.1s)

  Dataset:       dev.csv
  Mean score:    0.0161
  Solved rate:   2.0%
  Mean queries:  14.89 / 15
  Wall time:     0.1s


### Reference: a non-adaptive 20-questions sketch

This reference shows the *shape* of a real solution without giving away the points. In `__init__` it precomputes, with its own copy of the model, the oracle's yes/no answer to a small **fixed** list of broad questions for every animal — a bit-vector per animal. In `solve` it asks those same fixed questions, reads off the oracle's bit-vector, and guesses the animals whose precomputed vector is closest.

It works, but it's deliberately weak: the questions are the **same for every row** (not chosen adaptively to split the *remaining* candidates), and it uses only a handful. Beating it is mostly about (1) precomputing the full `animal × question` table and (2) choosing each next question *greedily* to most evenly split the animals still consistent with the answers so far. That's your job in Step 4.

> Precomputing even this small table calls the model a few thousand times (~5-15 min on T4). Skip this cell if you just want to get to your own solution — it is only a reference.

In [4]:
# Reference solution (optional, slow to init). Demonstrates precompute + match,
# but uses FIXED, non-adaptive questions -> leaves most of the score on the table.
FIXED_QUESTIONS = [
    'is it a mammal?',
    'is it a bird?',
    'is it a fish?',
    'is it an insect?',
    'does it live in water?',
    'can it fly?',
    'is it a carnivore?',
    'is it bigger than a human?',
    'does it have a backbone?',
    'is it commonly kept as a pet?',
    'does it have legs?',
    'does it lay eggs?',
]

class FixedQuestionsReference:
    def __init__(self, animals_pool, questions_pool, max_animals=None):
        self.animals_pool = animals_pool
        self.questions_pool = set(questions_pool)
        self.fixed = [q for q in FIXED_QUESTIONS if q in self.questions_pool]
        from interactor import Interactor
        cand = animals_pool if max_animals is None else animals_pool[:max_animals]
        self.candidates = cand
        print(f'  [reference] precomputing {len(cand)} x {len(self.fixed)} answer table...')
        self.table = {}
        for i, a in enumerate(cand):
            sim = Interactor(gold_animal=a, animals_pool=self.animals_pool,
                             questions_pool=self.questions_pool, budget=10**9)
            self.table[a] = tuple(1 if sim.ask(q) == 'yes' else 0 for q in self.fixed)
            if (i + 1) % 200 == 0:
                print(f'    {i+1}/{len(cand)}')
        print('  [reference] table ready.')

    def solve(self, interactor):
        obs = []
        for q in self.fixed:
            if interactor.remaining_budget() <= 1:
                break
            obs.append(1 if interactor.ask(q) == 'yes' else 0)
        obs = tuple(obs)
        def agree(a):
            vec = self.table[a]
            return sum(1 for x, y in zip(vec, obs) if x == y)
        ranked = sorted(self.candidates, key=agree, reverse=True)
        for a in ranked:
            if interactor.is_done():
                break
            interactor.guess(a)

# Example (commented out by default — uncomment to run; slow to init):
# ref = FixedQuestionsReference(animals_pool, questions_pool)
# ref_results = evaluate(ref, 'dev.csv')

## Step 4: Your solution

Replace the body of `MySolution.solve` (and `__init__` if you precompute anything) with your strategy. Iterate on `dev.csv` until you're happy with the score, then jump to Step 5 to evaluate on test1 + test2.

**The intended approach:**
1. In `__init__`, precompute once — with your own copy of the model — the oracle's yes/no answer for every `(animal, question)` pair you care about. This costs no oracle budget.
2. In `solve`, keep a set of candidate animals still consistent with the answers so far. At each step pick the **question whose answer most evenly splits that set** (maximize information gain), ask it, and shrink the set. Guess when one candidate dominates or the budget is nearly gone.
3. Be robust: the oracle occasionally answers in a way your table didn't predict. Don't let one surprising bit eliminate the true animal forever.

In [5]:
# =============================================================================
# IDEA 1 - the oracle is a DETERMINISTIC function, so tabulate it.
#   The judge is a greedy (temperature-0) LLM, so its answer to (animal, question)
#   is a fixed bit. We precompute those bits with our own copy of the model, which
#   costs no oracle budget. After that the game is 20-questions on a known table.
#
# IDEA 2 - we do not need all 559 questions.
#   The pool is ~10x redundant: 96 questions separate the animals as well as all
#   559 do (measured in simulation), which makes the table ~6x cheaper to build.
#
# IDEA 3 - ask what splits the posterior; guess when guessing is cheaper.
#   Keep a Bayesian posterior over the 1472 animals -- never hard-eliminate, since
#   the oracle answers from the model's beliefs and can surprise us, and one odd
#   answer must not destroy the true animal. Each turn, ask the question of highest
#   information gain, or guess the leader when guessing has the lower expected cost.
#
# SPEED - the table is 1472 x 96 = 141k prompts. Two tricks make that ~10 min on a T4:
#   * the judge names the animal BEFORE the question, so the ~52-token prefix is
#     shared by all questions of one animal: run it once, reuse its KV cache, and
#     each question then costs only its own ~12 tokens;
#   * never call generate(): a single forward pass gives the first token's logits,
#     and under greedy decoding the first token IS the answer.
# =============================================================================
import random
import time
from pathlib import Path

import numpy as np
import torch
from transformers import DynamicCache

from interactor import Interactor, JUDGE_PROMPT

N_QUESTIONS = 96      # runtime knob: ~2.3k tokens/animal -> ~10 min of T4.
                      # 96 and 128 and all-559 score the same (simulated); 64 does not.
EPS = 0.02            # assumed chance the oracle contradicts our table


def _hb(p):                                    # binary entropy, in bits
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))


def precompute_table(animals, questions, model, tok, chunk=128):
    """bits[a, q] = 1 iff the judge answers 'yes' about animal a for question q."""
    dev = model.device
    yes_t = torch.tensor([tok.encode(s)[0] for s in ('Yes', 'yes', 'YES')], device=dev)
    no_t  = torch.tensor([tok.encode(s)[0] for s in ('No', 'no', 'NO')], device=dev)

    def parts(animal, question):
        """Split the judge's prompt into (head ending at 'Question:', the rest).
        We cut BEFORE the space after 'Question:' -- BPE merges that space into the
        question's first token, so cutting after it would retokenize differently."""
        text = tok.apply_chat_template(
            [{'role': 'user',
              'content': JUDGE_PROMPT.format(animal=animal, question=question)}],
            tokenize=False, add_generation_prompt=True)
        head, rest = text.split('Question:')
        return head + 'Question:', rest

    # Question suffixes don't depend on the animal -> tokenize once. Group equal
    # lengths together so a batch needs no padding.
    suffixes = [tok(parts(animals[0], q)[1]).input_ids for q in questions]
    buckets = {}
    for qi, s in enumerate(suffixes):
        buckets.setdefault(len(s), []).append(qi)

    n_bkt = len(buckets)
    print(f'  [table] {len(animals)} animals x {len(questions)} questions '
          f'on {dev}  ({n_bkt} length-buckets, no padding)', flush=True)

    bits = np.zeros((len(animals), len(questions)), dtype=np.uint8)
    t0 = time.time()
    for ai, animal in enumerate(animals):
        head_ids = tok(parts(animal, '')[0]).input_ids
        n_pre = len(head_ids)
        with torch.inference_mode():                            # the shared prefix, once
            pre = model(torch.tensor([head_ids], device=dev),   # logits_to_keep=1:
                        use_cache=True, logits_to_keep=1)       # keep the KV, skip the LM head

        kv = [(l.keys, l.values) for l in pre.past_key_values.layers]

        for L, qis in buckets.items():
            for c in range(0, len(qis), chunk):
                grp = qis[c:c + chunk]
                B = len(grp)
                ids = torch.tensor([suffixes[qi] for qi in grp], device=dev)
                cache = DynamicCache([(k.expand(B, -1, -1, -1), v.expand(B, -1, -1, -1))
                                      for k, v in kv])          # every question reuses it
                with torch.inference_mode():
                    out = model(
                        input_ids=ids,
                        attention_mask=torch.ones((B, n_pre + L), dtype=torch.long, device=dev),
                        position_ids=torch.arange(n_pre, n_pre + L, device=dev).expand(B, L),
                        past_key_values=cache, logits_to_keep=1)
                lg = out.logits[:, -1, :].float()               # first-token logits
                yes = lg[:, yes_t].max(-1).values > lg[:, no_t].max(-1).values
                bits[ai, grp] = yes.cpu().numpy().astype(np.uint8)
        done = ai + 1
        if done % 100 == 0 or done == len(animals):
            el = time.time() - t0
            rate = done / el
            eta = (len(animals) - done) / rate
            print(f'    {done:>5}/{len(animals)} animals  '
                  f'{el:6.0f}s elapsed  {rate:5.1f} animals/s  '
                  f'eta {eta/60:4.1f} min', flush=True)
    print(f'  [table] done: {len(animals)} animals in {(time.time()-t0)/60:.1f} min '
          f'(yes-rate {bits.mean():.2f})', flush=True)
    return bits


class MySolution:
    def __init__(self, animals_pool, questions_pool, n_questions=N_QUESTIONS,
                 eps=EPS, alpha=1.4, seed=0):
        self.animals = list(animals_pool)
        # A random subset suffices -- the question pool is highly redundant.
        self.questions = sorted(random.Random(seed).sample(list(questions_pool), n_questions))

        # Cache the table on disk: __init__ runs again in the final scoring cell,
        # and rebuilding it there would just burn GPU minutes. The filename encodes
        # the pool size and we re-check the shape on load, so a cache built for a
        # different pool can never be used by mistake -- it is simply rebuilt.
        cache = Path(f'table_{len(self.animals)}x{n_questions}_{seed}.npy')
        self.bits = np.load(cache) if cache.exists() else None
        if self.bits is None or self.bits.shape != (len(self.animals), n_questions):
            Interactor._ensure_llm()                 # reuse the judge's own weights
            self.bits = precompute_table(self.animals, self.questions,
                                         Interactor._model, Interactor._tokenizer)
            np.save(cache, self.bits)
        self.alpha = alpha

        # P(oracle answers "yes" | hidden animal = a), allowing for a rare surprise.
        self.p_yes   = np.where(self.bits == 1, 1 - eps, eps).astype(np.float32)
        self.log_yes = np.log(self.p_yes)
        self.log_no  = np.log(1 - self.p_yes)
        self.h_cell  = _hb(self.p_yes).astype(np.float32)

    def solve(self, interactor):
        logp  = np.zeros(len(self.animals))          # uniform prior, in log space
        asked = np.zeros(len(self.questions), dtype=bool)

        while not interactor.is_done():
            p = np.exp(logp - logp.max())
            p /= p.sum()
            top = int(np.argmax(p))

            # Information gain of every question we have not asked yet.
            ig = _hb(p @ self.p_yes) - (p @ self.h_cell)
            ig[asked] = -1
            q = int(np.argmax(ig))

            if self._guess_is_better(p, top, float(ig[q]), interactor.remaining_budget()):
                if interactor.guess(self.animals[top]) == 'wrong':
                    logp[top] = -np.inf              # a wrong guess is exact information
            else:
                asked[q] = True
                ans = interactor.ask(self.questions[q])
                logp += (self.log_yes if ans == 'yes' else self.log_no)[:, q]

    def _guess_is_better(self, p, top, ig, budget):
        """A guess and a question both cost 0.02, so compare expected actions-to-go.
        With the surrogate C(p) = 1 + alpha*H(p) this reduces to the closed form
        below, which says: guess with 2-4 candidates left, keep asking with 5+."""
        if budget <= 1 or ig < 0.02:                 # must guess / nothing left to learn
            return True
        p_top = float(p[top])
        return p_top + self.alpha * _hb(p_top) > self.alpha * ig


my_dev = evaluate(MySolution(animals_pool, questions_pool), 'dev.csv')

  [table] 1472 animals x 96 questions on cuda:0  (8 length-buckets, no padding)


KeyboardInterrupt: 

## Step 5: Final scoring

Once you're happy with your dev score, run this cell. It scores `dev` and `test1` (and `test2` automatically, if that file is present). The **FINAL** line — the *n*-weighted mean over the available test split(s) — is what you submit a screenshot of.

> The organizers also score your submitted `MySolution` on a separate **hidden** test set that is not included here. Aim for a strategy that deduces the animal from scratch each row, so it transfers to unseen creatures.

In [ ]:
import os

solution = MySolution(animals_pool, questions_pool)
dev_results   = evaluate(solution, 'dev.csv')
test1_results = evaluate(solution, 'test1.csv')

splits = [('dev', dev_results), ('test1', test1_results)]
# test2 is a held-out set; included automatically only if present in the folder.
if os.path.exists('test2.csv'):
    splits.append(('test2', evaluate(solution, 'test2.csv')))

rows = [{
    'split': name, 'n': r['n'], 'mean_score': r['mean_score'],
    'solved_rate': r['solved_rate'], 'mean_queries': r['mean_queries'],
} for name, r in splits]

# FINAL = n-weighted mean over every test split available (test1 [+ test2]).
tests = [r for name, r in splits if name.startswith('test')]
n_test = sum(r['n'] for r in tests)
rows.append({
    'split': 'FINAL',
    'n': n_test,
    'mean_score':   sum(r['mean_score']   * r['n'] for r in tests) / n_test,
    'solved_rate':  sum(r['solved_rate']  * r['n'] for r in tests) / n_test,
    'mean_queries': sum(r['mean_queries'] * r['n'] for r in tests) / n_test,
})
pd.DataFrame(rows)

  [warn] row 0 (gold='aardwolf') raised: ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 30 is different from 1472)
  [warn] row 1 (gold='abalone') raised: ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 30 is different from 1472)
  [warn] row 2 (gold='african lion') raised: ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 30 is different from 1472)
  [warn] row 3 (gold='alpaca') raised: ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 30 is different from 1472)
  [warn] row 4 (gold='american black bear') raised: ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 30 is different from 1472)
  [warn] row 

,split,n,mean_score,solved_rate,mean_queries
0,dev,150,0.0,0.0,0.0
1,test1,500,0.0,0.0,0.0
2,FINAL,500,0.0,0.0,0.0
